# v0.3 · Notebook 05: Baselines

**LLM Security Gateway** · measures three simple detectors on the v0.2.1 dataset, so the fine-tuned model in notebook 06 has an honest target to beat.

| Detector | What it is | Why it's here |
|---|---|---|
| **Rules** | The gateway's own v0.1 rule layer, with its real block thresholds | What the gateway does today |
| **TF-IDF + logistic regression** | Counts words and character pieces, learns a weight for each | The classic cheap ML baseline: trains in about a minute, no GPU |
| **Open detector** | [`protectai/deberta-v3-base-prompt-injection-v2`](https://huggingface.co/protectai/deberta-v3-base-prompt-injection-v2), a popular open-source model | What someone would download instead of building their own |

**The rules of a fair comparison:**
- Learned detectors pick their block threshold on `validation` only: the lowest threshold that blocks **at most 1% of harmless messages** there. Test sets are never used to tune anything.
- Every detector is scored on the same five evaluation sets, broken down by language, source and hard benign text.

**Runtime:** GPU recommended (*Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU*) for the open detector. Everything else runs on CPU. Total time about 10 minutes.

In [ ]:
!pip -q install datasets

In [ ]:
import os, sys, json, time, subprocess
import numpy as np
import pandas as pd

CFG = {
    "dataset": "edithngalame/prompt-injection-en-es",
    "revision": "v0.2.1",    # dataset version (a tag on Hugging Face), so results are reproducible
    "open_model": "protectai/deberta-v3-base-prompt-injection-v2",
    "target_fpr": 0.01,      # block at most 1% of harmless validation messages
    "out_dir": "/content/v03_baselines",
    "repo": "https://github.com/edithngalame/llm-security-gateway",
    "repo_dir": "/content/llm-security-gateway",
}
RUN_OPEN_MODEL = True        # set to False to skip the open detector (e.g. no GPU)
EVAL_SETS = ["test", "test_heldout", "test_native_es", "test_other_lang", "eval_handwritten"]
pd.set_option("display.max_colwidth", 140)
os.makedirs(CFG["out_dir"], exist_ok=True)

In [ ]:
# Get the gateway's rule detector straight from your repo (it only needs the Python standard library)
if not os.path.exists(CFG["repo_dir"]):
    subprocess.run(["git", "clone", "-q", "--depth", "1", CFG["repo"], CFG["repo_dir"]], check=True)
sys.path.insert(0, f'{CFG["repo_dir"]}/src')

In [ ]:
from gateway.detectors import RuleDetector, Source
from gateway.policy import Action, Policy
import gateway
print("gateway version:", gateway.__version__)

## 1. Load the dataset from Hugging Face

Straight from your published dataset, pinned to the version in `CFG["revision"]`: no zip upload, no token needed now that it's public.

In [ ]:
from datasets import load_dataset
dd = load_dataset(CFG["dataset"], revision=CFG["revision"])
data = {name: dd[name].to_pandas() for name in dd}

In [ ]:
for name, d in data.items():
    print(f"{name:17s} {len(d):6,} rows  {int(d.label.sum()):6,} injections  languages: {', '.join(sorted(d.lang.unique())[:6])}")
# Does every split contain the same KINDS of text? If validation has no emails, a threshold
# tuned on it will not know how harmless emails score, and test FPR will come out higher.
is_email = lambda d: d.text.str.contains("SUBJECT:", regex=False)
mix = pd.DataFrame({s: {"harmless emails": int(((data[s].label == 0) & is_email(data[s])).sum()),
                        "email attacks": int(((data[s].label == 1) & is_email(data[s])).sum()),
                        "hard benign": int(data[s].family.fillna("").str.startswith("hard_benign").sum())}
                    for s in data}).T
display(mix)

## 2. How we score a detector

For every evaluation set we count:
- **Recall**: of the real attacks, how many were blocked. *Missed attacks are the security failure.*
- **False-positive rate (FPR)**: of the harmless messages, how many were wrongly blocked. *This is what makes users switch the gateway off.*
- **Precision**: of everything blocked, how much was really an attack.
- **PR-AUC**: one number summarising the detector at *every* threshold (1.0 = perfect). It's useful for comparing detectors without arguing about thresholds.

We also report them on **slices**: English vs Spanish, typed by a `user` vs hidden in a `retrieved` document, **hard benign** text (harmless but talks about rules or AI), and **harmless emails** (the leftover email bias from v0.2).

In [ ]:
from sklearn.metrics import average_precision_score

def rates(y, pred):
    y, pred = np.asarray(y), np.asarray(pred)
    tp = int(((y == 1) & (pred == 1)).sum()); fn = int(((y == 1) & (pred == 0)).sum())
    fp = int(((y == 0) & (pred == 1)).sum()); tn = int(((y == 0) & (pred == 0)).sum())
    return {"n": len(y), "attacks": tp + fn, "benign": fp + tn,
            "recall": tp / (tp + fn) if tp + fn else np.nan,
            "precision": tp / (tp + fp) if tp + fp else np.nan,
            "fpr": fp / (fp + tn) if fp + tn else np.nan}

def slices(df):
    # Named groups of rows to report on (a row can be in several)
    is_email = df.text.str.contains("SUBJECT:", regex=False)
    return {
        "all": np.ones(len(df), bool),
        "lang=en": (df.lang == "en").values,
        "lang=es": (df.lang == "es").values,
        "source=user": (df.source == "user").values,
        "source=retrieved": (df.source == "retrieved").values,
        "hard benign": ((df.label == 0) & df.family.fillna("").str.startswith("hard_benign")).values,
        "harmless emails": ((df.label == 0) & is_email).values,
    }

def pick_threshold(scores, y, target_fpr):
    # Lowest threshold that blocks at most target_fpr of the harmless messages
    benign = np.sort(np.asarray(scores)[np.asarray(y) == 0])[::-1]
    k = int(np.floor(target_fpr * len(benign)))       # how many harmless ones may score above it
    return float(np.nextafter(benign[k], 1.0)) if k < len(benign) else 0.0

DETECTORS = {}   # name -> {"scores": {set: array}, "preds": {set: array}, "threshold", "ms_per_text"}

def register(name, scores, preds, threshold, ms_per_text, note=""):
    DETECTORS[name] = {"scores": scores, "preds": preds, "threshold": threshold,
                       "ms_per_text": ms_per_text, "note": note}
    rows = []
    for s in EVAL_SETS:
        df, p = data[s], preds[s]
        for sl, mask in slices(df).items():
            if mask.sum():
                rows.append({"eval_set": s, "slice": sl, **rates(df.label.values[mask], p[mask])})
    t = pd.DataFrame(rows)
    main = t[t.slice == "all"].set_index("eval_set")[["n", "attacks", "recall", "fpr", "precision"]]
    print(f"{name}: threshold {threshold if isinstance(threshold, str) else f'{threshold:.3f}'}, "
          f"{ms_per_text:.2f} ms per text on CPU")
    display(main.style.format({"recall": "{:.1%}", "fpr": "{:.1%}", "precision": "{:.1%}"}, na_rep="–"))

def cpu_latency(fn, texts, n=200):
    # Average time to score ONE message at a time, like the gateway does
    texts = list(texts)[:n]
    t0 = time.perf_counter()
    for t in texts:
        fn([t])
    return (time.perf_counter() - t0) * 1000 / len(texts)

## 3. Baseline 1: the gateway's rules (v0.1)

Exactly what `/v1/scan` does today: rule score plus the per-source thresholds from `policy.py` (retrieved documents get a stricter threshold). Nothing is tuned here: this is the gateway as deployed.

In [ ]:
policy = Policy([RuleDetector()])

def rules_run(texts, sources=None):
    sources = sources if sources is not None else ["user"] * len(texts)
    dec = [policy.evaluate(t, Source(s) if s in ("user", "retrieved", "tool") else Source.USER)
           for t, s in zip(texts, sources)]
    return np.array([d.score for d in dec]), np.array([int(d.action == Action.BLOCK) for d in dec])

scores, preds = {}, {}
for s in EVAL_SETS:
    scores[s], preds[s] = rules_run(list(data[s].text), list(data[s].source))
register("Rules (v0.1)", scores, preds, "policy.py", cpu_latency(rules_run, data["test"].text))

**What to look for:** decent recall on English `test`, close to **0% on `test_native_es` and `test_other_lang`**: the rules only know English phrases. This is the gap the whole v0.3 release exists to close.

## 4. Baseline 2: TF-IDF + logistic regression

Two views of the text, glued together:
- **Words and word pairs** ("ignore previous", "system prompt")
- **Character pieces of 2–5 letters** ("ignor", "instru"): these survive typos, obfuscation and partly carry over between English and Spanish ("instru" is in *instructions* and *instrucciones*).

Logistic regression learns one weight per piece. `class_weight="balanced"` stops it from ignoring the rarer class (attacks are about 30% of training).

In [ ]:
from sklearn.pipeline import make_pipeline, make_union
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

tfidf = make_pipeline(
    make_union(
        TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, sublinear_tf=True, max_features=200_000),
        TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=3, sublinear_tf=True, max_features=300_000),
    ),
    LogisticRegression(C=4.0, class_weight="balanced", max_iter=3000),
)
train, val = data["train"], data["validation"]
t0 = time.time()
tfidf.fit(train.text, train.label)
print(f"trained on {len(train):,} texts in {time.time() - t0:.0f} s")

tfidf_score = lambda texts: tfidf.predict_proba(list(texts))[:, 1]
thr = pick_threshold(tfidf_score(val.text), val.label.values, CFG["target_fpr"])
val_pred = (tfidf_score(val.text) >= thr).astype(int)
print(f"threshold {thr:.3f} → on validation: " +
      ", ".join(f"{k} {v:.1%}" for k, v in rates(val.label, val_pred).items() if k in ("recall", "fpr")))

scores = {s: tfidf_score(data[s].text) for s in EVAL_SETS}
preds = {s: (scores[s] >= thr).astype(int) for s in EVAL_SETS}
register("TF-IDF + LR", scores, preds, thr, cpu_latency(tfidf_score, data["test"].text))

### What did it actually learn?

The heaviest weights show what the model relies on. **This is how you catch shortcut learning**: if the top "attack" pieces were things like `subject` or `.example`, the model would be detecting *emails*, not *attacks*.

In [ ]:
names = tfidf[0].get_feature_names_out()
names = [str(n).split("__", 1)[1] for n in names]      # drop the "which vectorizer" prefix
coef = tfidf[-1].coef_[0]
order = np.argsort(coef)
top = pd.DataFrame({
    "pushes towards ATTACK": [f"{names[i]!r}  ({coef[i]:+.1f})" for i in order[::-1][:25]],
    "pushes towards HARMLESS": [f"{names[i]!r}  ({coef[i]:+.1f})" for i in order[:25]],
})
display(top)

## 5. Baseline 3: an open-source detector

`protectai/deberta-v3-base-prompt-injection-v2` (Apache-2.0). Two things to know before reading its numbers:
- Its model card says it is **English only** and does **not target jailbreaks**. So low Spanish or role-play scores are expected, not a bug.
- It was trained on some public datasets, including `jackhhao/jailbreak-classification`, which is also one of **our** sources. It may have seen some of our test texts during its training, so its numbers could be **optimistic**.

For fairness it gets the same treatment as TF-IDF: threshold chosen on `validation` for a 1% false-positive budget. We also show how it does with its **shipped** threshold of 0.5.

Its score is the **logit margin** (how much more the model believes INJECTION than SAFE) rather than a probability. The model is so overconfident that many probabilities round to exactly 1.0, and a threshold can't separate tied scores. The margin ranks texts the same way, so PR-AUC is unchanged.

In [ ]:
if RUN_OPEN_MODEL:
    import torch
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("device:", device, "(GPU recommended)" if device == "cpu" else "")
    otok = AutoTokenizer.from_pretrained(CFG["open_model"])
    omodel = AutoModelForSequenceClassification.from_pretrained(CFG["open_model"]).to(device).eval()
    INJ = omodel.config.label2id["INJECTION"]

    @torch.no_grad()
    def open_score(texts, batch_size=32):
        texts, out = list(texts), []
        dev = next(omodel.parameters()).device
        for i in range(0, len(texts), batch_size):
            b = otok(texts[i:i + batch_size], return_tensors="pt", truncation=True, max_length=512, padding=True).to(dev)
            with torch.autocast("cuda", enabled=dev.type == "cuda"):
                logits = omodel(**b).logits
            # Score = logit margin (INJECTION minus SAFE) instead of a probability: this model is so
            # confident that many probabilities round to exactly 1.0, and tied scores make it
            # impossible to pick a threshold. The margin ranks texts the same way but never saturates.
            logits = logits.float()
            out.append((logits[:, INJ] - logits[:, 1 - INJ]).cpu().numpy().astype("float64"))
        return np.concatenate(out)

    t0 = time.time()
    val_s = open_score(val.text)
    thr = pick_threshold(val_s, val.label.values, CFG["target_fpr"])
    print(f"threshold {thr:.2f} (logit margin) → on validation: " +
          ", ".join(f"{k} {v:.1%}" for k, v in rates(val.label, (val_s >= thr).astype(int)).items() if k in ("recall", "fpr")))
    scores = {s: open_score(data[s].text) for s in EVAL_SETS}
    print(f"scored {len(val) + sum(len(data[s]) for s in EVAL_SETS):,} texts in {time.time() - t0:.0f} s")
    print(f"validation-tuned threshold {thr:.3f}")

    omodel.to("cpu")                                   # the gateway runs on CPU, so time it there
    ms = cpu_latency(open_score, data["test"].text, n=30)
    omodel.to(device)
    register("Open detector (ProtectAI)", scores,
             {s: (scores[s] >= thr).astype(int) for s in EVAL_SETS}, thr, ms)
    shipped = pd.DataFrame([{"eval_set": s, **rates(data[s].label, (scores[s] >= 0).astype(int))}
                            for s in EVAL_SETS]).set_index("eval_set")[["recall", "fpr"]]
    print("With its shipped threshold (probability 0.5, which is a logit margin of 0):")
    display(shipped.style.format("{:.1%}", na_rep="–"))

## 6. The scoreboard

One row per detector. **Test FPR, hard benign FPR and harmless email FPR should be low; everything else should be high.** This is the table that goes into the README.

*Hard benign* texts only exist in the hand-written sets (`eval_handwritten` and `test_native_es`, 26 texts), so that column pools them. It's small: read it as a warning light, not a precise number.

In [ ]:
def pick(name, s, sl, metric):
    # s = one eval set, or "ALL" to pool every eval set (used for the small hard benign slice)
    sets = EVAL_SETS if s == "ALL" else [s]
    y = np.concatenate([data[x].label.values[slices(data[x])[sl]] for x in sets])
    p = np.concatenate([DETECTORS[name]["preds"][x][slices(data[x])[sl]] for x in sets])
    return rates(y, p)[metric] if len(y) else np.nan

COLUMNS = {  # column name -> (eval set, slice, metric)
    "Recall: test": ("test", "all", "recall"),
    "FPR: test": ("test", "all", "fpr"),
    "Recall: Spanish (test)": ("test", "lang=es", "recall"),
    "Recall: human Spanish": ("test_native_es", "all", "recall"),
    "Recall: unseen styles": ("test_heldout", "all", "recall"),
    "Recall: other languages": ("test_other_lang", "all", "recall"),
    "Recall: hand-written": ("eval_handwritten", "all", "recall"),
    "FPR: hard benign": ("ALL", "hard benign", "fpr"),   # only the hand-written sets have them
    "FPR: harmless emails": ("test", "harmless emails", "fpr"),
}
board = pd.DataFrame({
    name: {**{c: pick(name, *spec) for c, spec in COLUMNS.items()},
           "PR-AUC: test": average_precision_score(data["test"].label, d["scores"]["test"]),
           "CPU ms/text": d["ms_per_text"]}
    for name, d in DETECTORS.items()}).T
pct = [c for c in board.columns if c.startswith(("Recall", "FPR"))]
display(board.style.format({**{c: "{:.1%}" for c in pct}, "PR-AUC: test": "{:.3f}", "CPU ms/text": "{:.1f}"}, na_rep="–"))

## 7. Charts

Left: attacks caught on each evaluation set. Right: harmless messages wrongly blocked (its scale is zoomed in, because these numbers are small). Saved as PNG for the README.

In [ ]:
import matplotlib.pyplot as plt

COLORS = ["#2a78d6", "#eb6834", "#1baf7a"]      # fixed order: rules, TF-IDF, open detector
INK, MUTED, GRID = "#1f1f1e", "#6b6a64", "#e6e5e0"

def grouped_bars(ax, groups, title, ymax=100):
    names = list(DETECTORS)
    x = np.arange(len(groups)); w = 0.8 / len(names)
    for i, name in enumerate(names):
        vals = [pick(name, *COLUMNS[g]) for g in groups]
        bars = ax.bar(x + (i - (len(names) - 1) / 2) * w, [0 if np.isnan(v) else v * 100 for v in vals],
                      w - 0.03, color=COLORS[i], label=name, zorder=2)
        for b, v in zip(bars, vals):
            ax.text(b.get_x() + b.get_width() / 2, b.get_height() + ymax * 0.015, "–" if np.isnan(v) else f"{v:.0%}",
                    ha="center", va="bottom", fontsize=8, color=MUTED)
    ax.set_xticks(x, [g.split(": ", 1)[1] for g in groups], fontsize=9, color=INK)
    ax.set_title(title, loc="left", fontsize=11, color=INK)
    ticks = np.linspace(0, ymax, 5)
    ax.set_ylim(0, ymax * 1.1); ax.set_yticks(ticks, [f"{v:g}%" for v in ticks], fontsize=8, color=MUTED)
    ax.grid(axis="y", color=GRID, zorder=0); ax.tick_params(length=0)
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)
    ax.spines["bottom"].set_color(GRID)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 4.2), gridspec_kw={"width_ratios": [2.3, 1]})
grouped_bars(a1, ["Recall: test", "Recall: Spanish (test)", "Recall: human Spanish", "Recall: unseen styles",
                  "Recall: other languages", "Recall: hand-written"], "Attacks caught (higher is better)")
fpr_cols = ["FPR: test", "FPR: hard benign", "FPR: harmless emails"]
worst = np.nanmax([pick(n, *COLUMNS[c]) for n in DETECTORS for c in fpr_cols] + [0.01]) * 100
grouped_bars(a2, fpr_cols, "Harmless messages blocked (lower is better)", ymax=max(10, 5 * np.ceil(worst / 5)))
fig.legend(*a1.get_legend_handles_labels(), frameon=False, fontsize=9, loc="upper left", ncol=len(DETECTORS), bbox_to_anchor=(0.01, 1.07))
fig.tight_layout()
fig.savefig(f'{CFG["out_dir"]}/baselines.png', dpi=160, bbox_inches="tight")
plt.show()

## 8. Look at the mistakes

Numbers tell you *how often*, examples tell you *why*. The most useful ones:
1. Attacks **every** detector missed: the hardest cases, and the ones your fine-tuned model most needs to learn.
2. Harmless messages the TF-IDF model blocked with the highest confidence: what it confuses with attacks.

In [ ]:
def show(df, cols=("text", "lang", "source", "family"), n=12):
    display(df.loc[:, list(cols)].head(n).assign(text=lambda d: d.text.str.slice(0, 220)))

for s in ["test_native_es", "test_heldout", "test"]:
    missed = np.logical_and.reduce([DETECTORS[n]["preds"][s] == 0 for n in DETECTORS]) & (data[s].label.values == 1)
    print(f"\n{s}: {missed.sum()} of {int(data[s].label.sum())} attacks missed by every detector")
    show(data[s][missed])

s = "test"
fp = (DETECTORS["TF-IDF + LR"]["preds"][s] == 1) & (data[s].label.values == 0)
print(f"\nHarmless messages in {s} blocked by TF-IDF: {fp.sum()} (highest scores first)")
show(data[s][fp].assign(score=DETECTORS["TF-IDF + LR"]["scores"][s][fp]).sort_values("score", ascending=False),
     cols=("text", "score", "lang", "family"))

## 9. Save the results

Everything goes into `v03_baselines_v0.2.1.zip` (named after the dataset version):
- `results.json`: scoreboard, thresholds and settings (goes into the repo)
- `results_by_slice.csv`: every eval set × slice × detector
- `scores.parquet`: every detector's score for every test text, so notebook 06 can compare against them without re-running this notebook
- `baselines.png`: the chart

In [ ]:
import zipfile

by_slice = []
for name, d in DETECTORS.items():
    for s in EVAL_SETS:
        for sl, mask in slices(data[s]).items():
            if mask.sum():
                by_slice.append({"detector": name, "eval_set": s, "slice": sl,
                                 **rates(data[s].label.values[mask], d["preds"][s][mask])})
pd.DataFrame(by_slice).to_csv(f'{CFG["out_dir"]}/results_by_slice.csv', index=False)

pd.concat([pd.DataFrame({"eval_set": s, "row": np.arange(len(data[s])), "label": data[s].label.values,
                         **{f"score::{n}": d["scores"][s] for n, d in DETECTORS.items()}})
           for s in EVAL_SETS]).to_parquet(f'{CFG["out_dir"]}/scores.parquet', index=False)

clean = lambda v: None if isinstance(v, float) and np.isnan(v) else round(float(v), 4)
results = {
    "release": "v0.3-baselines",
    "dataset": CFG["dataset"],
    "dataset_revision": CFG["revision"],
    "target_fpr_on_validation": CFG["target_fpr"],
    "eval_sizes": {s: len(data[s]) for s in EVAL_SETS},
    "detectors": {name: {"threshold": d["threshold"] if isinstance(d["threshold"], str) else round(d["threshold"], 4),
                         "scoreboard": {c: clean(v) for c, v in board.loc[name].items()}}
                  for name, d in DETECTORS.items()},
}
with open(f'{CFG["out_dir"]}/results.json', "w") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

ZIP = f'{"/content/" if os.path.exists("/content") else ""}v03_baselines_{CFG["revision"]}.zip'
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for fn in os.listdir(CFG["out_dir"]):
        z.write(f'{CFG["out_dir"]}/{fn}', fn)
print(json.dumps(results["detectors"], indent=1, ensure_ascii=False)[:1500])
print("\nsaved", ZIP)

In [ ]:
from google.colab import files
files.download(ZIP)

## 10. Questions to answer in your own words

Good interview practice. Write short answers before moving on.
1. Why do the rules score near 0% on Spanish, and why does TF-IDF do better even though most of its Spanish training data is machine-translated?
2. What happens to recall on `test_native_es` compared with the Spanish rows of `test`? What does that tell you about machine-translated training data?
3. Look at the top TF-IDF features. Do you see any shortcut (a feature that marks the *format* rather than the *attack*)?
4. Why is the threshold picked on `validation` and not on `test`?
5. The open detector may have seen some of our test data during its training. Would that make its numbers better or worse than they really are?
6. Which single number from the scoreboard would you show a security manager, and which one a product manager?